# 1. Check xem file test_question_window_sequences.csv không dùng is_repeat cũng có thể dự đoán all-in-one. Theo review 2026-10-10 nói rằng file test_question_window_sequences.csv, mỗi KC thứ hai cùng câu hỏi sẽ được cắt xuống dòng mới với cùng lịch sử và mask hết lịch sử chỉ chừa ra KC vừa cắt

In [1]:
import pandas as pd
 
df = pd.read_csv("/kaggle/input/datasets/nhatanhadr/xes3g5m/test_question_window_sequences.csv")
n_scored = n_leak = n_scored_repeat = n_pos0 = 0
for sm_s, qi_s, rep_s in zip(df["selectmasks"], df["qidxs"], df["is_repeat"]):
    sm = [int(x) for x in str(sm_s).split(",")]
    qi = [int(x) for x in str(qi_s).split(",")]
    rep = [int(x) for x in str(rep_s).split(",")]
    n_pos0 += sm[0] == 1
    for t, m in enumerate(sm):
        if m == 1:
            n_scored += 1
            n_leak += qi[t] in qi[:t]          # lịch sử có KC khác của cùng câu hỏi?
            n_scored_repeat += rep[t] == 1      # vị trí được chấm có is_repeat = 1 (M1)
print(n_scored, n_leak, n_scored_repeat, n_pos0)
# Kỳ vọng: n_leak == 0 (không rò rỉ ở test), n_pos0 == 0, n_scored_repeat > 0 (xác nhận M1)


1270391 0 171194 0


# 2. Check xem với qidxs sau khi lọc mask gom theo qidxs có đủ rest hay không

In [2]:
import pandas as pd
from collections import defaultdict

def parse_seq(s):
    return [int(x) for x in str(s).strip().split(",")]

qidx2rest = defaultdict(list)

for qidxs_s, masks_s, rest_s in zip(df["qidxs"], df["selectmasks"], df["rest"]):
    qidxs, masks, rests = parse_seq(qidxs_s), parse_seq(masks_s), parse_seq(rest_s)
    for q, m, r in zip(qidxs, masks, rests):
        if m == 1:
            qidx2rest[q].append(r)

def is_valid(values):
    return sorted(values) == list(range(len(values)))

invalid = {q: v for q, v in qidx2rest.items() if not is_valid(v)}

print(f"Tổng số qidx: {len(qidx2rest)}")
print(f"Số qidx lỗi: {len(invalid)}")
for q, v in list(invalid.items())[:10]:
    print(q, sorted(v))

Tổng số qidx: 1099197
Số qidx lỗi: 0


# 3. Cột qidxs được mô tả là mỗi câu hỏi được trả lời tại một thời điểm có index duy nhất, check xem có thực sự là nó duy nhất không, nghĩa là với mỗi qidxs ứng với một học sinh trả lời câu hỏi đó, tại một thời điểm qidxs -> (uid, question, timestamp)

In [3]:
df_raw = df

cols = ['questions', 'timestamps', 'qidxs', 'selectmasks']
keep = ['qidxs', 'uid', 'questions', 'timestamps', 'selectmasks']
parts = []

for i in range(0, len(df_raw), 50_000):
    d = df_raw.iloc[i:i + 50_000][['uid'] + cols].copy()
    for c in cols:
        d[c] = d[c].astype(str).str.split(',')

    lens = d[cols].apply(lambda s: s.str.len())
    assert (lens.nunique(axis=1) == 1).all(), f"Lệch độ dài ở chunk {i}"

    d = d.explode(cols).astype('int64')
    d = d[d['qidxs'] >= 0]
    parts.append(d[keep].drop_duplicates()) 

u = pd.concat(parts).drop_duplicates()

key = ['uid', 'timestamps', 'questions']

# Chiều 1: qidx -> tương tác
chk = u.groupby('qidxs')[['uid', 'questions', 'timestamps']].nunique()
print("Số qidx:", len(chk))
print("qidx có >1 uid     :", (chk.uid > 1).sum())
print("qidx có >1 question:", (chk.questions > 1).sum())
print("qidx có >1 ts      :", (chk.timestamps > 1).sum())

Số qidx: 1102810
qidx có >1 uid     : 0
qidx có >1 question: 0
qidx có >1 ts      : 0
